# Results section 4, and panel e of Figure 3

## Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# the package root is the first folder at or above this one that holds Data/
def _root(start=None):
    here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().resolve()
    for c in (here, *here.parents):
        if (c / "Data" / "conversation_turns.csv").exists():
            return c
    raise RuntimeError(f"cannot find the package root (no Data/conversation_turns.csv at or above {here})")

ROOT   = _root()
CLEAN  = ROOT / "Data"
RESDIR = ROOT / "Code" / "_output" / "results"; RESDIR.mkdir(parents=True, exist_ok=True)

### Helper functions

In [2]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

def glmm(formula, data, crossed, label=""):
    # crossed random intercepts, variational Bayes
    vc = {g: f"0 + C({g})" for g in crossed}
    # seed the variational fit
    np.random.seed(2026)
    m = BinomialBayesMixedGLM.from_formula(formula, vc, data).fit_vb()
    if label:
        print(f"  {label}")
    print(f"    {'term':<18}{'b':>9}{'SD':>8}{'95% CrI':>21}{'odds':>8}")
    for i, nm in enumerate(m.model.exog_names):
        lo, hi = m.fe_mean[i] - 1.96 * m.fe_sd[i], m.fe_mean[i] + 1.96 * m.fe_sd[i]
        print(f"    {nm:<18}{m.fe_mean[i]:>9.3f}{m.fe_sd[i]:>8.3f}"
              f"   [{lo:+.3f}, {hi:+.3f}]{np.exp(m.fe_mean[i]):>8.2f}")
    print("    random intercepts (SD, logit scale): "
          + "  ".join(f"{n} {np.exp(v):.2f}" for n, v in zip(m.model.vcp_names, m.vcp_mean)))
    return m

def robust_logit(formula, data, cluster):
    return smf.logit(formula, data=data).fit(disp=0, cov_type="cluster",
                                             cov_kwds={"groups": data[cluster]})

def contrast(fit, a, b):
    # Wald test on the difference between two coefficients
    names = list(fit.params.index)
    R_ = np.zeros(len(names)); R_[names.index(a)] = 1; R_[names.index(b)] = -1
    diff = fit.params[a] - fit.params[b]
    se = float(np.sqrt(R_ @ fit.cov_params().values @ R_))
    w = fit.wald_test(R_.reshape(1, -1), scalar=True)
    print(f"    {a} - {b} = {diff:+.3f}   SE = {se:.3f}   "
          f"95% CI [{diff - 1.96*se:+.3f}, {diff + 1.96*se:+.3f}]   "
          f"chi2 = {float(w.statistic):.2f}, p = {float(w.pvalue):.2e}")
    return diff, se

## 1 · Stores and patches

In [3]:
STIMULI = pd.read_csv(CLEAN / "stimuli.csv")
_st = STIMULI[~STIMULI.is_lure]
ID2CAT = {int(i): int(l) for i, l in zip(_st.item_id, _st.list)}
RECALL = pd.read_csv(CLEAN / "individual_recall.csv")
pre = RECALL[(RECALL.phase == "pre") & (RECALL.kind == "studied")].sort_values(
    ["participant", "output_position"])
PRIVATE = {str(p): [int(i) for i in g.item_id] for p, g in pre.groupby("participant")}
# private recall as item-id sets by category; a store is a set minus words already written by either participant
OWN_IDS = {}
for pp, v in PRIVATE.items():
    OWN_IDS[pp] = {}
    for i in v:
        OWN_IDS[pp].setdefault(ID2CAT[int(i)], set()).add(int(i))

def store(person, cat, ids, cats, upto):
    # words of `cat` that `person` held before the conversations and nobody has written in turns
    # 0 .. upto-1
    eaten = {ids[u] for u in range(upto) if cats[u] == cat and ids[u] is not None}
    return len(OWN_IDS.get(str(person), {}).get(cat, set()) - eaten)

STREAM = pd.read_pickle(RESDIR / "stream.pkl")

def patch_blocks(cats, kinds):
    blocks, cur = [], []
    for i, c in enumerate(cats):
        if c is None:
            if cur:
                blocks.append(cur); cur = []
            continue
        if cur and cats[cur[-1]] == c:
            cur.append(i)
        else:
            if cur:
                blocks.append(cur)
            cur = [i]
    if cur:
        blocks.append(cur)
    return blocks

## 2 · Whose store predicts continuation (Results section 3; Supplementary Note 5.3)

In [4]:
rows = []
for _, r in STREAM.iterrows():
    cats, kinds, spk, pid = r.cats, r.kinds, r.speakers, r.pid
    n = len(cats)
    for b in patch_blocks(cats, kinds):
        cat = cats[b[0]]
        for k in range(1, len(b) + 1):
            last = b[k - 1]; cur_sp = spk[last]
            # censor the last word and turns followed by the same participant
            if last + 1 >= n:
                continue
            nxt_sp = spk[last + 1]
            if nxt_sp == cur_sp:
                continue

            rows.append(dict(conversation=f"{r.network}_{r.dyad}", network=str(r.network),
                             category=str(cat), depth=k, continues=int(len(b) > k),
                             speaker=str(pid[nxt_sp]), other=str(pid[cur_sp]),
                             spk_tray=store(pid[nxt_sp], cat, r.words, cats, last + 1),
                             oth_tray=store(pid[cur_sp], cat, r.words, cats, last + 1),
                             how_late=b[0] / max(n - 1, 1)))
STEPS = pd.DataFrame(rows)
STEPS["dcap"] = STEPS.depth.clip(upper=4).astype(str)
for c_ in ("spk_tray", "oth_tray"):
    STEPS["z_" + c_] = (STEPS[c_] - STEPS[c_].mean()) / STEPS[c_].std()

In [5]:
_ = glmm("continues ~ z_spk_tray + z_oth_tray + C(dcap) + how_late", STEPS,
         ["speaker", "other", "conversation", "category", "network"],
         "random intercepts: speaker, other, conversation, category, network")

  random intercepts: speaker, other, conversation, category, network
    term                      b      SD              95% CrI    odds
    Intercept            -1.573   0.026   [-1.625, -1.522]    0.21
    C(dcap)[T.2]          1.579   0.061   [+1.460, +1.698]    4.85
    C(dcap)[T.3]          1.332   0.083   [+1.170, +1.495]    3.79
    C(dcap)[T.4]          1.305   0.077   [+1.154, +1.456]    3.69
    z_spk_tray            0.671   0.025   [+0.622, +0.721]    1.96
    z_oth_tray            0.069   0.026   [+0.018, +0.120]    1.07
    how_late              0.093   0.051   [-0.008, +0.194]    1.10
    random intercepts (SD, logit scale): speaker 0.71  other 0.09  conversation 0.10  category 0.37  network 0.06


In [6]:
mh = robust_logit("continues ~ z_spk_tray + z_oth_tray + C(dcap) + how_late",
                  STEPS, "conversation")
print("  speaker store minus other's store, SEs clustered on the conversation")
CONTRAST_H = contrast(mh, "z_spk_tray", "z_oth_tray")

nets = STEPS.network.unique()
rng = np.random.default_rng(7); boot = []
for _ in range(2000):
    pick = rng.choice(nets, len(nets), replace=True)
    d = pd.concat([STEPS[STEPS.network == nn].assign(_b=f"{nn}_{ii}")
                   for ii, nn in enumerate(pick)])
    try:
        mb = smf.logit("continues ~ z_spk_tray + z_oth_tray + C(dcap) + how_late",
                       d).fit(disp=0)
        boot.append(mb.params["z_spk_tray"] - mb.params["z_oth_tray"])
    except Exception:
        pass
boot = np.array(boot)
print(f"  network bootstrap, speaker - other {boot.mean():+.3f}   95% CI [{np.percentile(boot, 2.5):+.3f}, "
      f"{np.percentile(boot, 97.5):+.3f}]")

  speaker store minus other's store, SEs clustered on the conversation
    z_spk_tray - z_oth_tray = +0.494   SE = 0.045   95% CI [+0.406, +0.581]   chi2 = 122.71, p = 1.61e-28
  network bootstrap, speaker - other +0.495   95% CI [+0.406, +0.591]


## 3 · Offered category vs the responder's own category at the first handover

In [7]:
excl = []
rows = []
for _, r in STREAM.iterrows():
    cats, ids, spk, pid = r.cats, r.words, r.speakers, r.pid
    cid = f"{r.network}_{r.dyad}"
    for b in patch_blocks(cats, r.kinds):
        t = b[0]; opener = spk[t]; cat = cats[t]
        resp = [x for x in pid if x != opener][0]
        me = str(pid[resp])
        prev = [u for u in range(t) if spk[u] == resp and cats[u] is not None]
        if not prev:
            continue                                  # nothing of their own in play yet
        own_cat = cats[prev[-1]]
        excl.append(dict(person=me, same=int(own_cat == cat)))
        if own_cat == cat:
            continue                                  # own category is the offered one
        eaten_own = {ids[u] for u in range(t) if cats[u] == own_cat}
        eaten_off = {ids[u] for u in range(t + 1) if cats[u] == cat}
        # unanswered handovers (the answering turn is not the responder's) are censored
        if not (t + 1 < len(cats) and spk[t + 1] == resp):
            excl[-1]["censored"] = 1
            continue
        rows.append(dict(conversation=cid, network=str(r.network), person=me,
                         nth=int(r.nth[resp]), category=str(cat),
                         own_left=len(OWN_IDS.get(me, {}).get(own_cat, set()) - eaten_own),
                         off_left=len(OWN_IDS.get(me, {}).get(cat, set()) - eaten_off),
                         took_it_up=int(len(b) > 1)))
WEIGH_ALL = pd.DataFrame(rows)
# the model excludes handovers with no words left in the responder's own category
WEIGH = WEIGH_ALL[WEIGH_ALL.own_left > 0].copy().reset_index(drop=True)
for c_ in ("off_left", "own_left"):
    WEIGH["z_" + c_] = (WEIGH[c_] - WEIGH[c_].mean()) / WEIGH[c_].std()

# exclusions, as a share of each participant's first handovers
_z = WEIGH_ALL.assign(z=WEIGH_ALL.own_left == 0).groupby("person").z.mean()
print(f"  excluded, no words left in own category: "
      f"{100 * _z.mean():.1f}% per participant (SD {100 * _z.std(ddof=1):.1f})")
_ex = pd.DataFrame(excl).fillna({"censored": 0})
_g = _ex.groupby("person").agg(n=("same", "size"), same=("same", "sum"),
                               cens=("censored", "sum"))
_g["pct_cens"] = 100 * _g["cens"] / (_g["n"] - _g["same"])
print(f"  excluded, no reply: {_g.pct_cens.mean():.1f}% per participant (SD {_g.pct_cens.std():.1f})")

# Results section 3: declined first handovers with words of the offered category left
_away = WEIGH_ALL[WEIGH_ALL.took_it_up == 0]
_per = _away.groupby("person").apply(lambda g: (g.off_left > 0).mean())
print(f"  declined with offered-category words left: {100 * _per.mean():.1f}% "
      f"(SD {100 * _per.std(ddof=1):.1f}) over {len(_per)} participants")

  excluded, no words left in own category: 26.8% per participant (SD 17.7)
  excluded, no reply: 3.3% per participant (SD 3.7)
  declined with offered-category words left: 56.6% (SD 26.0) over 170 participants


### 3.1 · Uptake model (Results section 4; Supplementary Note 1)

In [8]:
_ = glmm("took_it_up ~ z_off_left + z_own_left", WEIGH,
         ["person", "category", "conversation", "network"],
         "random intercepts: person, category, conversation, network")
# person means of each store added (Mundlak)
for c_ in ("off_left", "own_left"):
    _pm = WEIGH.groupby("person")[c_].transform("mean")
    WEIGH["pm_" + c_] = (_pm - _pm.mean()) / _pm.std()
mw = robust_logit("took_it_up ~ z_off_left + z_own_left + pm_off_left + pm_own_left",
                  WEIGH, "person")
print("  within-participant model, clustered on the participant")
for t in ("z_off_left", "z_own_left"):
    print(f"    {t:<12}{mw.params[t]:+.3f}  SE {mw.bse[t]:.3f}   p = {mw.pvalues[t]:.1e}")
rngw = np.random.default_rng(3); nets = WEIGH.network.unique(); bt = []
for _ in range(2000):
    d = pd.concat([WEIGH[WEIGH.network == n] for n in rngw.choice(nets, len(nets), replace=True)])
    try:
        bt.append(smf.logit("took_it_up ~ z_off_left + z_own_left",
                            d).fit(disp=0).params["z_own_left"])
    except Exception:
        pass
bt = np.array(bt)
print(f"  network bootstrap, z_own_left {bt.mean():+.3f}   95% CI [{np.percentile(bt, 2.5):+.3f}, "
      f"{np.percentile(bt, 97.5):+.3f}]")

  random intercepts: person, category, conversation, network
    term                      b      SD              95% CrI    odds
    Intercept            -1.663   0.042   [-1.746, -1.580]    0.19
    z_off_left            0.559   0.039   [+0.483, +0.635]    1.75
    z_own_left           -0.157   0.044   [-0.244, -0.071]    0.85
    random intercepts (SD, logit scale): person 0.89  category 0.48  conversation 0.14  network 0.09
  within-participant model, clustered on the participant
    z_off_left  +0.508  SE 0.045   p = 3.5e-29
    z_own_left  -0.106  SE 0.040   p = 8.0e-03
  network bootstrap, z_own_left -0.162   95% CI [-0.244, -0.087]


### 3.2 · Uptake model in each conversation (Supplementary Note 5.1)

In [9]:
(RESDIR / "supp").mkdir(exist_ok=True)
_bn = []
for _k in (1, 2, 3):
    _d = WEIGH[WEIGH.nth == _k]
    _m = robust_logit("took_it_up ~ z_off_left + z_own_left", _d, "conversation")
    for _t, _lab in (("z_off_left", "the offer"), ("z_own_left", "their own")):
        _lo, _hi = _m.conf_int().loc[_t]
        _bn.append(dict(nth=_k, term=_lab, b=_m.params[_t], lo=_lo, hi=_hi, n=len(_d)))
pd.DataFrame(_bn).to_csv(RESDIR / "supp" / "s_bynth_contest.csv", index=False)
print("  offered vs own store, by conversation")
print(pd.DataFrame(_bn).round(3).to_string(index=False))

  offered vs own store, by conversation
 nth      term      b     lo     hi    n
   1 the offer  0.667  0.525  0.809 1299
   1 their own -0.230 -0.391 -0.068 1299
   2 the offer  0.458  0.332  0.584 1451
   2 their own -0.132 -0.260 -0.005 1451
   3 the offer  0.430  0.310  0.550 1468
   3 their own -0.132 -0.263 -0.001 1468


### 3.3 · Equal and opposite weights (Results section 4; Supplementary Note 5.1)

In [10]:
_aic = {}
for f_, lab in (("took_it_up ~ z_off_left + z_own_left", "both, free to differ"),
                ("took_it_up ~ I(off_left - own_left)", "the difference alone")):
    _aic[lab] = smf.logit(f_, WEIGH).fit(disp=0).aic
print(f"  delta AIC, difference model minus free model: "
      f"{_aic['the difference alone'] - _aic['both, free to differ']:.2f}")
# per word: test b_offered + b_own = 0, SEs clustered on participant
mpw = robust_logit("took_it_up ~ off_left + own_left", WEIGH, "person")
_R = np.zeros(len(mpw.params)); _ix = list(mpw.params.index)
_R[_ix.index("off_left")] = 1; _R[_ix.index("own_left")] = 1
_s = mpw.params["off_left"] + mpw.params["own_left"]
_se = float(np.sqrt(_R @ mpw.cov_params().values @ _R))
_w = mpw.wald_test(_R.reshape(1, -1), scalar=True)
print(f"  per word: offered + own = {_s:+.3f}, 95% CI [{_s - 1.96*_se:+.3f}, {_s + 1.96*_se:+.3f}], "
      f"chi2(1) = {float(_w.statistic):.2f}, p = {float(_w.pvalue):.1e}")
for _k in (1, 2, 3):
    _m = robust_logit("took_it_up ~ off_left + own_left", WEIGH[WEIGH.nth == _k], "conversation")
    _s = _m.params["off_left"] + _m.params["own_left"]
    _se = float(np.sqrt(_R @ _m.cov_params().values @ _R))
    print(f"    conversation {_k}: offered + own = {_s:+.3f}, "
          f"95% CI [{_s - 1.96*_se:+.3f}, {_s + 1.96*_se:+.3f}]")

  delta AIC, difference model minus free model: 27.38
  per word: offered + own = +0.171, 95% CI [+0.084, +0.257], chi2(1) = 14.84, p = 1.2e-04
    conversation 1: offered + own = +0.212, 95% CI [+0.107, +0.317]
    conversation 2: offered + own = +0.162, 95% CI [+0.050, +0.274]
    conversation 3: offered + own = +0.147, 95% CI [+0.027, +0.268]


## 4 · Data for Figure 3e

In [11]:
rgb = np.random.default_rng(31)

def binned(df, col, y, cap=4, cluster="conversation", nboot=2000):
    # bin means with a bootstrap over the clustering unit
    out = []
    d = df.assign(b=df[col].clip(upper=cap))
    units = d[cluster].unique()
    for k in sorted(d.b.unique()):
        sub = d[d.b == k]
        # per-cluster sums and counts
        agg = sub.groupby(cluster)[y].agg(["sum", "size"]).reindex(units, fill_value=0)
        S_, N_ = agg["sum"].values, agg["size"].values
        draws = []
        for _ in range(nboot):
            ix = rgb.integers(0, len(units), len(units))
            if N_[ix].sum():
                draws.append(S_[ix].sum() / N_[ix].sum())
        out.append((k, sub[y].mean(), np.percentile(draws, 2.5),
                    np.percentile(draws, 97.5), len(sub)))
    return pd.DataFrame(out, columns=["b", "m", "lo", "hi", "n"])

# binned uptake values written out for Figure 3e (read by results3_first_handover_cost.ipynb)
(RESDIR / "supp").mkdir(exist_ok=True)
rows = []
for sel, lab in [(WEIGH_ALL.own_left == 0, "none left"),
                 (WEIGH_ALL.own_left >= 1, "still holding")]:
    g = binned(WEIGH_ALL[sel], "off_left", "took_it_up", cap=3, cluster="conversation")
    rows.append(g.assign(own=lab))
UPTAKE = pd.concat(rows, ignore_index=True)
UPTAKE.to_csv(RESDIR / "supp" / "s_uptake_by_store.csv", index=False)
print("uptake by store (Figure 3e)")
print(UPTAKE.round(3).to_string(index=False))

uptake by store (Figure 3e)
 b     m    lo    hi    n           own
 0 0.113 0.090 0.137  762     none left
 1 0.236 0.186 0.291  284     none left
 2 0.320 0.245 0.393  169     none left
 3 0.466 0.409 0.529  296     none left
 0 0.090 0.076 0.105 1534 still holding
 1 0.178 0.147 0.208  732 still holding
 2 0.246 0.214 0.282  619 still holding
 3 0.313 0.282 0.345 1333 still holding
